In [ ]:
# 环境自检:打印本 notebook 依赖的第三方库版本,便于排查环境问题
from importlib.metadata import version

# pkgs 列表列出了运行该 notebook(下载/加载 Tiny Aya 权重、构建模型)所需的关键依赖包
pkgs = [
    #"blobfile",         # to download pretrained weights
    "huggingface_hub",  # to download pretrained weights
    # 注意(风险项):本 notebook 实际使用的是下面 TinyAyaTokenizer 中的 `tokenizers` 库(Tokenizer.from_file)完成分词,
    # 并未在任何地方 import/使用 tiktoken;这里检查 tiktoken 版本很可能是从其他笔记本(如 Llama 系列)复制遗留下来的,
    # 若当前环境未安装 tiktoken,下面的 version(p) 会直接报错(PackageNotFoundError)。是否移除/替换为 "tokenizers" 建议人工确认后再改,此处不擅自改动。
    "tiktoken",         # to implement the tokenizer
    "torch",            # to implement the model
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
from pathlib import Path

# 选择要下载/加载的 Tiny Aya 权重仓库;官方发布了 4 个变体(global/fire/water/earth),取消注释切换即可
REPO_ID = "CohereLabs/tiny-aya-global"
#REPO_ID = "CohereLabs/tiny-aya-fire"
#REPO_ID = "CohereLabs/tiny-aya-water"
#REPO_ID = "CohereLabs/tiny-aya-earth"
# 本地缓存目录名取仓库名的最后一段(如 "tiny-aya-global"),用于存放下载的分词器与权重文件

LOCAL_DIR = Path(REPO_ID).parts[-1]

In [ ]:
# ============================================================
# Tiny Aya (Cohere2 架构) 模型组件定义
# 整体是类似 Llama 的 Decoder-only Transformer,但有几个关键差异:
#   1) 归一化层是「无 bias 的 LayerNorm」(CohereLayerNorm),而不是 RMSNorm;
#   2) 使用分组查询注意力 GQA(Grouped Query Attention)节省 KV 缓存;
#   3) RoPE 采用「奇偶交错(interleaved)」布局,且只在滑动窗口(局部)注意力层生效,
#      全局(full_attention)层不加位置编码(NoPE),二者按固定比例交替出现;
#   4) TransformerBlock 是「并行残差结构」:注意力分支和前馈分支共享同一个输入归一化结果,
#      两者的输出直接与残差相加,而不是像标准 Transformer 那样先做注意力再做前馈的串行结构。
# ============================================================
import torch
import torch.nn as nn



# FeedForward:门控前馈网络(SwiGLU 变体)
# 输入 x 形状: (batch_size, seq_len, emb_dim)
# fc1、fc2 都把 emb_dim 投影到 hidden_dim(分别对应 HF 权重中的 gate_proj / up_proj),
# 用 SiLU 激活 fc1 的输出后与 fc2 的输出逐元素相乘(门控),再由 fc3(down_proj)投影回 emb_dim
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)

    def forward(self, x):
        # x_fc1、x_fc2 形状均为 (batch_size, seq_len, hidden_dim)
        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x)
        x = nn.functional.silu(x_fc1) * x_fc2
        return self.fc3(x)
# Aya uses a bias-less LayerNorm variant.
# The difference to classic LayerNorm is that it only
# has a scale parameter (weight), no shift parameter (bias).

# 中文说明:这里的 CohereLayerNorm 并不是严格意义上的 RMSNorm。
# RMSNorm 只用「均方根」做缩放,不做去均值(mean-centering);
# 而这里仍然计算了 mean 并做了 (x - mean),本质上是标准 LayerNorm 去掉 bias、只保留 weight(scale)参数的变体。
class CohereLayerNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(emb_dim))

    def forward(self, x):
        input_dtype = x.dtype
        x = x.to(torch.float32)
        mean = x.mean(dim=-1, keepdim=True)
        variance = (x - mean).pow(2).mean(dim=-1, keepdim=True)
        x = (x - mean) * torch.rsqrt(variance + self.eps)
        return (self.weight.to(torch.float32) * x).to(input_dtype)

# ---------------- RoPE(旋转位置编码) ----------------
# compute_rope_params: 预计算所有位置的 cos/sin 表
# head_dim: 每个注意力头的维度;theta_base: RoPE 的基数 theta(此模型用 50000,而非常见的 10000);
# context_length: 预计算的最大位置数(此模型配置里高达 500_000,因此该表会很大,注意显存/内存开销)
def compute_rope_params(head_dim, theta_base=10_000, context_length=4096, dtype=torch.float32):
    assert head_dim % 2 == 0, "head_dim must be even"

    # Compute the inverse frequencies
    inv_freq = 1.0 / (
        theta_base ** (torch.arange(0, head_dim, 2, dtype=dtype)[: (head_dim // 2)].float() / head_dim)
    )
    positions = torch.arange(context_length, dtype=dtype)

    # Compute the angles
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)  # Shape: (context_length, head_dim // 2)

    # Cohere uses interleaved even/odd angle layout per head-dim pair.
    # Llama2 notebook examples often use a split-halves layout via cat([angles, angles]).
    # Both are equivalent only when paired with the matching rotate logic:
    # - interleaved layout -> even/odd rotation implementation (below)
    # - split-halves layout -> half/half rotate implementation
    angles = torch.repeat_interleave(angles, 2, dim=1)  # Shape: (context_length, head_dim)

    # Precompute sine and cosine
    # 返回的 cos、sin 形状均为 (context_length, head_dim),后续会按当前序列长度切片使用
    return torch.cos(angles), torch.sin(angles)


# apply_rope: 将 RoPE 施加到 Q 或 K 上(奇偶交错布局:偶数维和奇数维两两一组做旋转)
def apply_rope(x, cos, sin):
    # x: (batch_size, num_heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "head_dim must be even"

    # Split x into even and odd components (interleaved layout)
    x_even = x[..., ::2]
    x_odd = x[..., 1::2]

    # Adjust sin and cos shapes
    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)

    # Apply the rotary transformation
    # 旋转公式: x_rotated = x * cos + rotate(x) * sin,其中 rotate 对相邻的偶/奇维度做 (-odd, even) 交换
    x_float = x.float()
    rotated = torch.stack((-x_odd.float(), x_even.float()), dim=-1).flatten(-2)
    x_rotated = (x_float * cos) + (rotated * sin)

    return x_rotated.to(dtype=x.dtype)

# ---------------- 分组查询注意力 GQA ----------------
# num_heads 个 Query 头共享 num_kv_groups 组 Key/Value 头(num_heads 必须整除 num_kv_groups),
# 每组被 group_size = num_heads // num_kv_groups 个 Query 头复用,从而大幅减少 KV 缓存显存占用。
# 例如本模型 n_heads=16, n_kv_heads=4 => group_size=4,即每 4 个 Query 头共用 1 组 K/V。
class GroupedQueryAttention(nn.Module):
    def __init__(
        self,
        d_in,
        num_heads,
        num_kv_groups,
        head_dim=None,
        qk_norm=False,
        attention_bias=False,
        dtype=None,
        attn_type="full_attention",
    ):
        super().__init__()
        assert num_heads % num_kv_groups == 0, "num_heads must be divisible by num_kv_groups"

        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups

        if head_dim is None:
            assert d_in % num_heads == 0, "`d_in` must be divisible by `num_heads` if `head_dim` is not set"
            head_dim = d_in // num_heads

        self.head_dim = head_dim
        self.d_out = num_heads * head_dim
        self.attn_type = attn_type

        # W_query: emb_dim -> num_heads * head_dim(每个 token 产生 num_heads 个 query 向量)
        self.W_query = nn.Linear(
            d_in,
            self.d_out,
            bias=attention_bias,
            dtype=dtype,
        )
        # W_key / W_value: emb_dim -> num_kv_groups * head_dim(K、V 头数远少于 Q 头数,这就是 GQA 省显存的关键)
        self.W_key = nn.Linear(
            d_in,
            num_kv_groups * head_dim,
            bias=attention_bias,
            dtype=dtype,
        )
        self.W_value = nn.Linear(
            d_in,
            num_kv_groups * head_dim,
            bias=attention_bias,
            dtype=dtype,
        )
        self.out_proj = nn.Linear(
            self.d_out,
            d_in,
            bias=attention_bias,
            dtype=dtype,
        )

        if qk_norm:
            self.q_norm = CohereLayerNorm(head_dim, eps=1e-6)
            self.k_norm = CohereLayerNorm(head_dim, eps=1e-6)
        else:
            self.q_norm = self.k_norm = None

    # forward 输入 x: (batch_size, num_tokens, d_in);mask 为布尔张量,True 表示需要屏蔽(mask 掉)的位置
    def forward(self, x, mask, cos, sin):
        b, num_tokens, _ = x.shape

        # Apply projections
        queries = self.W_query(x)  # (b, num_tokens, num_heads * head_dim)
        keys = self.W_key(x)       # (b, num_tokens, num_kv_groups * head_dim)
        values = self.W_value(x)   # (b, num_tokens, num_kv_groups * head_dim)

        # Reshape
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys = keys.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)
        values = values.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)

        # 变换后: queries 形状 (b, num_heads, num_tokens, head_dim);keys/values 形状 (b, num_kv_groups, num_tokens, head_dim)
        # Optional normalization
        if self.q_norm:
            queries = self.q_norm(queries)
        if self.k_norm:
            keys = self.k_norm(keys)

        # Cohere applies RoPE only on sliding-attention layers.
        # 这正是 Cohere2 混合架构的关键设计:滑窗(局部)层用 RoPE 编码相对位置,
        # 全局(full_attention)层依赖前面局部层已经混入的位置信息,不再重复施加位置编码(NoPE)
        if self.attn_type == "sliding_attention":
            queries = apply_rope(queries, cos, sin)
            keys = apply_rope(keys, cos, sin)

        # Expand K and V to match number of heads
        # repeat_interleave 把每个 KV 头复制 group_size 次,使 K/V 的头数与 Q 对齐,才能逐头做点积注意力
        keys = keys.repeat_interleave(self.group_size, dim=1)
        values = values.repeat_interleave(self.group_size, dim=1)

        # Attention
        # attn_scores 形状: (b, num_heads, num_tokens, num_tokens)
        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores = attn_scores.masked_fill(mask, -torch.inf)

        # 按 sqrt(head_dim) 缩放后做 softmax,防止点积数值过大导致梯度/数值不稳定
        attn_weights = torch.softmax(attn_scores / self.head_dim**0.5, dim=-1)
        # context 变换回 (b, num_tokens, d_out),d_out = num_heads * head_dim,再经 out_proj 投影回 d_in 维度
        context = (attn_weights @ values).transpose(1, 2).reshape(b, num_tokens, self.d_out)

        return self.out_proj(context)

# ---------------- Transformer 块(并行残差结构) ----------------
# 注意:Cohere/Aya 的 Block 只有一个 input_layernorm,注意力分支和前馈分支都读取同一份归一化结果 x,
# 二者的输出与原始残差 shortcut 三者相加,而不是像 GPT/Llama 那样「norm1->attn->残差->norm2->ffn->残差」的串行结构
class TransformerBlock(nn.Module):
    def __init__(self, cfg, attn_type):
        super().__init__()
        self.attn_type = attn_type

        # 注意:这里显式传入 qk_norm=False,即该 Tiny Aya 配置未启用 Query/Key 归一化(与部分 Cohere 变体不同)
        self.att = GroupedQueryAttention(
            d_in=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            num_kv_groups=cfg["n_kv_heads"],
            head_dim=cfg["head_dim"],
            qk_norm=False,
            attention_bias=cfg["attention_bias"],
            dtype=cfg["dtype"],
            attn_type=attn_type,
        )
        self.ff = FeedForward(cfg)
        self.input_layernorm = CohereLayerNorm(cfg["emb_dim"], eps=cfg["layer_norm_eps"])

    # 根据当前层是滑窗层还是全局层,选择对应的注意力 mask(mask_local 或 mask_global)
    def forward(self, x, mask_global, mask_local, cos, sin):
        attn_mask = mask_local if self.attn_type == "sliding_attention" else mask_global

        shortcut = x
        x = self.input_layernorm(x)
        x_attn = self.att(x, attn_mask, cos, sin)  # Shape [batch_size, num_tokens, emb_dim]
        x_ff = self.ff(x)

        # Cohere parallel residual block
        x = shortcut + x_attn + x_ff
        return x

# ---------------- 完整模型:TinyAyaModel ----------------
# 组成: token embedding -> N 层 TransformerBlock(按 layer_types 交替 sliding/full attention)
# -> 最终归一化 -> 输出头(可与输入 embedding 权重绑定) -> 乘以 logit_scale 得到最终 logits
class TinyAyaModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert len(cfg["layer_types"]) == cfg["n_layers"], "layer_types must match n_layers"

        self.cfg = cfg

        # 词嵌入表: 形状 (vocab_size, emb_dim)
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])
        self.trf_blocks = nn.ModuleList([TransformerBlock(cfg, t) for t in cfg["layer_types"]])

        self.final_norm = CohereLayerNorm(cfg["emb_dim"], eps=cfg["layer_norm_eps"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

        self.logit_scale = cfg["logit_scale"]

        # 预先计算好整段 context_length 的 RoPE cos/sin 表并注册为非持久 buffer(不会被存进 state_dict)
        cos, sin = compute_rope_params(
            head_dim=cfg["head_dim"],
            theta_base=cfg["rope_base"],
            context_length=cfg["context_length"],
        )
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

        # 权重绑定(weight tying):输出头与输入词嵌入共享同一份参数矩阵,减少参数量
        if cfg["tie_word_embeddings"]:
            self.out_head.weight = self.tok_emb.weight


    # 构建两种注意力 mask:
    #   mask_global: 标准因果(下三角)mask,只允许看到当前及之前的 token
    #   mask_local : 在因果 mask 基础上再叠加「滑动窗口」限制,只能看到最近 sliding_window 个 token
    # 两个 mask 中 True 都表示「需要被屏蔽」的位置
    def create_masks(self, num_tokens, device):
        ones = torch.ones((num_tokens, num_tokens), dtype=torch.bool, device=device)

        # Future mask
        mask_global = torch.triu(ones, diagonal=1)

        # Sliding-window mask
        far_past = torch.triu(ones, diagonal=self.cfg["sliding_window"]).T
        mask_local = mask_global | far_past

        # Expand to [batch, heads, seq, seq]-broadcastable shape
        return mask_global.unsqueeze(0).unsqueeze(0), mask_local.unsqueeze(0).unsqueeze(0)


    # 模型前向:input_ids 形状 (batch_size, num_tokens) -> 输出 logits 形状 (batch_size, num_tokens, vocab_size)
    def forward(self, input_ids, attention_mask=None):
        tok_embeds = self.tok_emb(input_ids)
        x = tok_embeds
        num_tokens = input_ids.shape[1]

        mask_global, mask_local = self.create_masks(num_tokens, x.device)

        # 若提供了 padding mask,则把被 padding 的位置也叠加进两种注意力 mask 中
        if attention_mask is not None:
            # True means mask in this implementation.
            pad_mask = attention_mask[:, None, None, :].to(dtype=torch.bool).logical_not()
            mask_global = mask_global | pad_mask
            mask_local = mask_local | pad_mask

        # 按当前实际序列长度截取预计算好的 RoPE 表
        cos = self.cos[:num_tokens, :].to(x.device, dtype=x.dtype)
        sin = self.sin[:num_tokens, :].to(x.device, dtype=x.dtype)

        # 依次通过每一层 TransformerBlock,每层根据自己的 attn_type 选择 mask_global 或 mask_local
        for block in self.trf_blocks:
            x = block(x, mask_global, mask_local, cos, sin)

        x = self.final_norm(x)
        # Cohere 系列会对最终 logits 做一次缩放(logit_scale),再返回给上层做 softmax/采样
        logits = self.out_head(x.to(self.cfg["dtype"]))
        return logits * self.logit_scale

2. Initialize model
The remainder of this notebook uses the Llama 3.2 1B model; to use the 3B model variant, just uncomment the second configuration file in the following code cell

In [ ]:
# Tiny Aya 模型的超参数配置(对应 HuggingFace Cohere2 config.json 中的字段)
TINY_AYA_CONFIG = {
    "vocab_size": 262_144,            # Vocabulary size
    # 注意:context_length 高达 500_000,会用于预计算 RoPE 的 cos/sin 表(见 compute_rope_params),因此该表本身会占用不小的内存
    "context_length": 500_000,        # Context length in the HF config
    "emb_dim": 2048,                  # Embedding dimension
    "n_heads": 16,                    # Number of attention heads
    "n_layers": 36,                   # Number of layers
    "hidden_dim": 11_008,             # Size of the intermediate dimension in FeedForward
    # 这里 head_dim=128 恰好等于 emb_dim/n_heads=2048/16,说明 head_dim 是显式配置而非从二者推导得出
    "head_dim": 128,                  # Size of the heads in GQA
    # n_kv_heads=4 < n_heads=16 => 采用分组查询注意力(GQA),每 4 个 Query 头共享 1 组 K/V,大幅节省推理时的 KV 缓存
    "n_kv_heads": 4,                  # Number of KV heads for grouped-query attention
    "attention_bias": False,          # Whether attention projections use bias terms
    "attention_dropout": 0.0,         # Attention dropout
    "sliding_window": 4096,           # Sliding-window attention context
    # layer_types 决定每一层是滑动窗口局部注意力还是全局注意力;下面 36 层按「3 个 sliding + 1 个 full」循环 9 次
    "layer_types": [
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
        "sliding_attention",
        "sliding_attention",
        "sliding_attention",
        "full_attention",
    ],
    # 注意 rope_base=50_000,比很多模型默认的 10_000 更大,这样可以让 RoPE 编码更长的上下文
    "rope_base": 50_000.0,            # The base in RoPE's "theta"
    "layer_norm_eps": 1e-5,           # Epsilon used by layer normalization
    "logit_scale": 1.0,               # Final logits scaling factor
    "tie_word_embeddings": True,      # Whether input embedding and output head are tied
    "bos_token_id": 2,
    "eos_token_id": 3,
    "pad_token_id": 0,
    "dtype": torch.bfloat16,          # Lower-precision dtype to reduce memory usage
}

# 用上面的配置实例化模型(此时权重是随机初始化的,后续单元会加载预训练权重)
model = TinyAyaModel(TINY_AYA_CONFIG)

# 估算模型在给定 dtype 下大致会占用多少显存/内存(仅统计参数 + 梯度占位 + buffer,不含激活值)
def calc_model_memory_size(model, input_dtype=torch.float32):
    total_params = 0
    total_grads = 0
    for param in model.parameters():
        # Calculate total number of elements per parameter
        param_size = param.numel()
        total_params += param_size
        # Check if gradients are stored for this parameter
        if param.requires_grad:
            total_grads += param_size

    # Calculate buffer size (non-parameters that require memory)
    total_buffers = sum(buf.numel() for buf in model.buffers())

    # Size in bytes = (Number of elements) * (Size of each element in bytes)
    # We assume parameters and gradients are stored in the same type as input dtype
    element_size = torch.tensor(0, dtype=input_dtype).element_size()
    total_memory_bytes = (total_params + total_grads + total_buffers) * element_size

    # Convert bytes to gigabytes
    total_memory_gb = total_memory_bytes / (1024**3)

    return total_memory_gb

print(f"float32 (PyTorch default): {calc_model_memory_size(model, input_dtype=torch.float32):.2f} GB")
print(f"bfloat16: {calc_model_memory_size(model, input_dtype=torch.bfloat16):.2f} GB")

In [ ]:
# 统计模型总参数量;由于 tie_word_embeddings=True,输出头权重与词嵌入权重是同一份参数,
# 会被 model.parameters() 重复计入,因此下面减去一份 tok_emb 的参数量得到「去重后」的真实参数量
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

# Account for weight tying
total_params_normalized = total_params - model.tok_emb.weight.numel()
print(f"\nTotal number of unique parameters: {total_params_normalized:,}")

In [ ]:
# 自动选择可用的计算设备:优先 CUDA GPU,其次 Apple Silicon 的 MPS,最后回退到 CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# 把模型搬到目标设备上(分号只是抑制 Jupyter 输出该表达式的返回值)
model.to(device);

3. Load tokenizer

In [ ]:
# ---------------- 分词器 ----------------
# 直接使用 HuggingFace 的 `tokenizers` 库加载 fast tokenizer 的 tokenizer.json 文件(不依赖 transformers)
from tokenizers import Tokenizer


# 对 encode/decode 做一层轻量封装,并自动从 tokenizer.json 里读取真实的特殊 token id(找不到时退回传入的默认值)
class TinyAyaTokenizer:
    def __init__(self, tokenizer_file_path, eos_token_id=3, pad_token_id=0, bos_token_id=2):
        tok_file = Path(tokenizer_file_path)
        self._tok = Tokenizer.from_file(str(tok_file))

        # 优先使用 tokenizer.json 内置的特殊符号 id;若该分词器没有定义,则使用构造函数传入的默认值兜底
        eos_from_tok = self._tok.token_to_id("<EOS_TOKEN>")
        pad_from_tok = self._tok.token_to_id("<PAD>")
        bos_from_tok = self._tok.token_to_id("<BOS_TOKEN>")

        self.eos_token_id = eos_from_tok if eos_from_tok is not None else eos_token_id
        self.pad_token_id = pad_from_tok if pad_from_tok is not None else pad_token_id
        self.bos_token_id = bos_from_tok if bos_from_tok is not None else bos_token_id

    def encode(self, text):
        return self._tok.encode(text).ids

    def decode(self, ids):
        return self._tok.decode(ids, skip_special_tokens=False)



# 按 Cohere/Aya 的对话模板拼接 prompt(特殊 token 标记对话轮次的开始/结束、角色等)
def apply_chat_template(user_text):
    return (
        "<BOS_TOKEN>"
        "<|START_OF_TURN_TOKEN|><|USER_TOKEN|>"
        f"{user_text}"
        "<|END_OF_TURN_TOKEN|>"
        "<|START_OF_TURN_TOKEN|><|CHATBOT_TOKEN|><|START_RESPONSE|>"
    )

In [ ]:
# 打印当前选用的 REPO_ID,确认后续下载/加载用的是哪个 Tiny Aya 变体
print(REPO_ID)

In [ ]:
# ---------------- 下载分词器并登录 HuggingFace Hub ----------------
# Uncomment and run the following code if you are executing the notebook for the first time

# 首次运行需要登录(部分仓库可能是 gated),交互式输入 HF token
from huggingface_hub import login
login()
from huggingface_hub import hf_hub_download

# 若本地还没有 tokenizer.json,则从 HF Hub 下载到 LOCAL_DIR;下载失败时给出警告并退回一个占位路径
tokenizer_file_path = Path(LOCAL_DIR) / "tokenizer.json"
if not tokenizer_file_path.exists():
    try:
        tokenizer_file_path = hf_hub_download(repo_id=REPO_ID, filename="tokenizer.json", local_dir=LOCAL_DIR)
    except Exception as e:
        print(f"Warning: failed to download tokenizer.json: {e}")
        tokenizer_file_path = "tokenizer.json"

# 用下载好的 tokenizer.json 以及配置里的特殊 token id 构造分词器
tokenizer = TinyAyaTokenizer(
    tokenizer_file_path=Path(LOCAL_DIR) / "tokenizer.json",
    eos_token_id=TINY_AYA_CONFIG["eos_token_id"],
    pad_token_id=TINY_AYA_CONFIG["pad_token_id"],
    bos_token_id=TINY_AYA_CONFIG["bos_token_id"],
)


# 用聊天模板包装一个示例问题,再编码成 token id,然后立刻解码回文本,验证分词器工作正常(含特殊 token 一起还原)
prompt = apply_chat_template("Give me a short introduction to large language models in 3 sentences.")
input_token_ids = tokenizer.encode(prompt)
text = tokenizer.decode(input_token_ids)
text

4. Load pretrained weights

In [ ]:
# ---------------- 加载预训练权重 ----------------
# 把从 HF safetensors 里读到的 state_dict(键名遵循 Cohere2ForCausalLM 的命名规范)逐一拷贝进我们自定义模块的参数里
def load_weights_into_tiny_aya(model, param_config, params):
    # assign: 先检查形状是否一致,再用 copy_ 原地把预训练权重写入目标参数(保留目标参数原有的 dtype/device)
    def assign(left, right, tensor_name="unknown"):
        if left.shape != right.shape:
            raise ValueError(
                f"Shape mismatch in tensor '{tensor_name}'. Left: {left.shape}, Right: {right.shape}"
            )

        with torch.no_grad():
            if isinstance(right, torch.Tensor):
                left.copy_(right.to(dtype=left.dtype, device=left.device))
            else:
                left.copy_(torch.as_tensor(right, dtype=left.dtype, device=left.device))

        return left

    # 词嵌入表: model.embed_tokens.weight -> tok_emb.weight
    model.tok_emb.weight = assign(
        model.tok_emb.weight,
        params["model.embed_tokens.weight"],
        "model.embed_tokens.weight",
    )


    # 逐层拷贝每个 TransformerBlock 里的注意力投影、前馈网络投影、层归一化权重
    for l in range(param_config["n_layers"]):
        block = model.trf_blocks[l]
        att = block.att

        # Q, K, V projections
        att.W_query.weight = assign(
            att.W_query.weight,
            params[f"model.layers.{l}.self_attn.q_proj.weight"],
            f"model.layers.{l}.self_attn.q_proj.weight",
        )
        att.W_key.weight = assign(
            att.W_key.weight,
            params[f"model.layers.{l}.self_attn.k_proj.weight"],
            f"model.layers.{l}.self_attn.k_proj.weight",
        )
        att.W_value.weight = assign(
            att.W_value.weight,
            params[f"model.layers.{l}.self_attn.v_proj.weight"],
            f"model.layers.{l}.self_attn.v_proj.weight",
        )

        # Output projection
        att.out_proj.weight = assign(
            att.out_proj.weight,
            params[f"model.layers.{l}.self_attn.o_proj.weight"],
            f"model.layers.{l}.self_attn.o_proj.weight",
        )

        # Feedforward weights
        block.ff.fc1.weight = assign(
            block.ff.fc1.weight,
            params[f"model.layers.{l}.mlp.gate_proj.weight"],
            f"model.layers.{l}.mlp.gate_proj.weight",
        )
        block.ff.fc2.weight = assign(
            block.ff.fc2.weight,
            params[f"model.layers.{l}.mlp.up_proj.weight"],
            f"model.layers.{l}.mlp.up_proj.weight",
        )
        block.ff.fc3.weight = assign(
            block.ff.fc3.weight,
            params[f"model.layers.{l}.mlp.down_proj.weight"],
            f"model.layers.{l}.mlp.down_proj.weight",
        )

        # Layernorm
        block.input_layernorm.weight = assign(
            block.input_layernorm.weight,
            params[f"model.layers.{l}.input_layernorm.weight"],
            f"model.layers.{l}.input_layernorm.weight",
        )

    # Final normalization and output head
    model.final_norm.weight = assign(
        model.final_norm.weight,
        params["model.norm.weight"],
        "model.norm.weight",
    )


    # 若权重文件里单独提供了 lm_head.weight 就直接加载它;否则回退为权重绑定(输出头复用词嵌入权重)
    if "lm_head.weight" in params:
        model.out_head.weight = assign(model.out_head.weight, params["lm_head.weight"], "lm_head.weight")
    else:
        if param_config["tie_word_embeddings"]:
            model.out_head.weight = model.tok_emb.weight
            print("Model uses weight tying.")

# 实际下载权重并调用上面的 load_weights_into_tiny_aya:
# 1) 用 index.json 找出权重被切分成了哪些 safetensors 分片文件
# 2) 依次加载每个分片并合并成一个完整的 state_dict
# 3) 灌入模型后释放这份临时 state_dict 以节省内存
import json
from safetensors.torch import load_file
from huggingface_hub import snapshot_download


# 下载整个仓库快照(权重分片 + index 文件等)到本地
repo_dir = snapshot_download(repo_id=REPO_ID, local_dir=LOCAL_DIR)
index_path = Path(repo_dir) / "model.safetensors.index.json"
with open(index_path, "r") as f:
    index = json.load(f)

# 按 index.json 里记录的分片文件名,逐个加载 safetensors 分片并合并进同一个字典
weights_dict = {}
for filename in sorted(set(index["weight_map"].values())):
    shard_path = Path(repo_dir) / filename
    shard = load_file(shard_path)
    weights_dict.update(shard)


# 把合并好的权重写入模型,再搬到目标设备,并删除巨大的临时字典释放内存
load_weights_into_tiny_aya(model, TINY_AYA_CONFIG, weights_dict)
model.to(device)
del weights_dict

In [ ]:
# 加载权重后再次统计「去重」参数量:通过 data_ptr() 判断哪些参数张量在内存里是同一份(比如被权重绑定共享的词嵌入/输出头),
# 避免重复计数,用来和前面 cell 5 估算的理论去重参数量做交叉验证
def count_unique_parameters(model):
    unique_params = set()
    total_unique_params = 0

    for param in model.parameters():
        if param.data_ptr() not in unique_params:
            total_unique_params += param.numel()
            unique_params.add(param.data_ptr())

    return total_unique_params

total_params_uniq = count_unique_parameters(model)
print(f"Total number of unique parameters: {total_params_uniq:,}")

5. Generate text

In [ ]:
# ---------------- 文本生成 ----------------
# 收集所有可能表示「回复结束」的 token id(EOS、以及 Aya 对话模板里的 <|END_RESPONSE|> / <|END_OF_TURN_TOKEN|>)
stop_ids = {
    tokenizer.eos_token_id,
    tokenizer._tok.token_to_id("<|END_RESPONSE|>"),
    tokenizer._tok.token_to_id("<|END_OF_TURN_TOKEN|>"),
}
# 过滤掉分词器里找不到、返回 None 的 token(即该特殊符号在这份词表里不存在)
stop_ids = {x for x in stop_ids if x is not None}



# 最基础的贪心自回归生成:每步都对整个已生成序列重新做一次完整前向传播(没有 KV 缓存),
# 取最后一个位置 logits 里概率最大的 token 作为下一个 token,遇到停止符就提前结束;用生成器 yield 逐个吐出新 token
def generate_text_basic_stream(model, token_ids, max_new_tokens, stop_token_ids=None):
    stop_token_ids = set(stop_token_ids or [])

    model.eval()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            # out 形状: (batch_size=1, vocab_size),只取序列最后一个位置的 logits 用于预测下一个 token
            out = model(token_ids)[:, -1]
            next_token = torch.argmax(out, dim=-1, keepdim=True)

            # batch size 1
            if next_token.item() in stop_token_ids:
                break

            yield next_token
            token_ids = torch.cat([token_ids, next_token], dim=1)

# 用真实 prompt 跑一遍流式生成,并在支持 CUDA 时统计峰值显存占用
prompt = apply_chat_template("Give me a short introduction to large language models in 3 sentences.")
input_token_ids = tokenizer.encode(prompt)
# unsqueeze(0) 增加 batch 维度,得到形状 (1, prompt_len) 的输入张量
input_token_ids_tensor = torch.tensor(input_token_ids, device=device).unsqueeze(0)


if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()



# 逐个 token 解码并立即打印,实现打字机式的流式输出效果
for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=500,
    stop_token_ids=stop_ids
):
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )


# 打印生成过程中 GPU 显存的峰值占用,便于评估该模型的推理显存开销
if torch.cuda.is_available():
    def calc_gpu_gb(x):
        return f"{x / 1024 / 1024 / 1024:.2f} GB"

    print(f"\n\nGPU memory used: {calc_gpu_gb(torch.cuda.max_memory_allocated())}")